In [ ]:
import pandas as pd
import numpy as np
np.set_printoptions(legacy="1.25")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

#!wget $data -O data-week-3.csv

In [ ]:
df = pd.read_csv('data-week-3.csv')
df

In [ ]:
df.head().T

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ','_')

In [ ]:
df.dtypes

In [ ]:
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [ ]:
df.head().T

In [ ]:
#pd.to_numeric(df.totalcharges)

In [ ]:
tc = pd.to_numeric(df.totalcharges, errors = 'coerce')

In [ ]:
df[tc.isnull()][['customerid', 'totalcharges']]

In [ ]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors = 'coerce')

In [ ]:
df.totalcharges = df.totalcharges.fillna(0)

In [ ]:
df.churn.head()

In [ ]:
(df.churn == 'yes').astype(int).head

In [ ]:
df.churn = (df.churn == 'yes').astype(int)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_test_split?

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [ ]:
len(df_full_train), len(df_test)

In [ ]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [ ]:
len(df_train), len(df_val), len(df_test)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [ ]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [ ]:
df_full_train = df_full_train.reset_index(drop=True)

In [ ]:
df_full_train.head()

In [ ]:
df_full_train.isnull().sum()

In [ ]:
df_full_train.churn.value_counts(normalize=True)

In [ ]:
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate,2)

In [ ]:
df_full_train.dtypes

In [ ]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [ ]:
df_full_train.columns

In [ ]:
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
        'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [ ]:
df_full_train[categorical].nunique()

In [ ]:
df_full_train.churn.mean()

In [ ]:
global_churn = df_full_train.churn.mean()
global_churn

In [ ]:
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_female, churn_male

In [ ]:
global_churn - churn_female

In [ ]:
global_churn - churn_male

In [ ]:
df_full_train.partner.value_counts()

In [ ]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

In [ ]:
churn_partner, churn_no_partner

In [ ]:
global_churn - churn_partner

In [ ]:
global_churn - churn_no_partner

In [ ]:
churn_no_partner/global_churn

In [ ]:
churn_partner/global_churn

In [ ]:
df_full_train.groupby('gender').churn.mean()

In [ ]:
df_group = df_full_train.groupby('gender').churn.agg(['count', 'mean'])

In [ ]:
df_group['diff'] = df_group['mean'] - global_churn
df_group['risk'] = df_group['mean']/global_churn
df_group

In [ ]:
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['count', 'mean'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean']/global_churn
    print(df_group)
    print()
    print()
    

In [ ]:
from sklearn.metrics import mutual_info_score

In [ ]:
mutual_info_score(df_full_train.churn, df_full_train.onlinesecurity)

In [ ]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [ ]:
df_full_train[categorical].apply(mutual_info_churn_score)

In [ ]:
pd.DataFrame.apply?

In [ ]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

In [ ]:
numerical

In [ ]:
df_full_train[numerical].corrwith(df_full_train.churn)

In [ ]:
df_full_train[numerical].corrwith(df_full_train.churn).abs()

In [ ]:
df_full_train[df_full_train['tenure'] <= 2].churn.mean()

In [ ]:
df_full_train[(df_full_train['tenure'] > 2) & (df_full_train['tenure'] <= 12)].churn.mean()

In [ ]:
df_full_train[df_full_train['tenure'] > 12].churn.mean()

In [ ]:
df_full_train[df_full_train['monthlycharges'] <= 20].churn.mean()

In [ ]:
df_full_train[(df_full_train['monthlycharges'] > 20) & (df_full_train['monthlycharges'] <= 50)].churn.mean()

In [ ]:
df_full_train[df_full_train['monthlycharges'] > 50].churn.mean()

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
df_train[['gender', 'contract']].iloc[:10].to_dict()

In [ ]:
df_train[['gender', 'contract']].iloc[:100].to_dict(orient='records')

In [ ]:
dicts = df_train[['gender', 'contract']].iloc[:100].to_dict(orient='records')

In [ ]:
dv = DictVectorizer(sparse=False)

In [ ]:
dv.fit(dicts)

In [ ]:
dv.get_feature_names_out()

In [ ]:
dv.transform(dicts)

In [ ]:
train_dicts = df_train[categorical+numerical].to_dict(orient='records')

In [ ]:
train_dicts[0]

In [ ]:
dv.fit(train_dicts)

In [ ]:
dv.get_feature_names_out()

In [ ]:
list(dv.transform(train_dicts[:5])[0])

In [ ]:
X_train = dv.fit_transform(train_dicts)

In [ ]:
X_train.shape

In [ ]:
val_dicts = df_val[categorical+numerical].to_dict(orient='records')

In [ ]:
X_val = dv.transform(val_dicts)

In [ ]:
X_val.shape

In [ ]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

In [ ]:
z = np.linspace(5, -5, 51)

In [ ]:
z

In [ ]:
sigmoid(z)

In [ ]:
plt.plot(z, sigmoid(z))

In [ ]:
plt.plot?

In [ ]:
def linear_regression(xi):
    n = len(xi)
    
    result = w0
    
    for j in range(n):
        result = result + w[j] * xi[j]
    return result

In [ ]:
def logistic_regression(xi):
    n = len(xi)
    
    score = w0
    
    for j in range(n):
        score = score + w[j] * xi[j]

    result = sigmoid(score)
    return result

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
model.coef_[0].round(3)

In [ ]:
model.intercept_[0]

Hard Predictions

In [ ]:
model.predict(X_train)

Soft Predictions
[probability of 0, probability of 1]

In [ ]:
model.predict_proba(X_train)

In [ ]:
y_pred = model.predict_proba(X_val)[:,1]

In [ ]:
y_pred >= 0.5

In [ ]:
churn_decision = (y_pred >= 0.5)

In [ ]:
df_val[churn_decision]

In [ ]:
df_val[churn_decision].customerid

In [ ]:
y_val == churn_decision

In [ ]:
(y_val == churn_decision).mean()

In [ ]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = churn_decision.astype(int)
df_pred['actual'] = y_val


In [ ]:
df_pred

In [ ]:
df_pred['correct'] = df_pred.prediction == df_pred.actual

In [ ]:
df_pred

In [ ]:
df_pred.correct.mean()

Evaluation

In [ ]:
len(y_val)

In [ ]:
(y_val == churn_decision).sum()

In [ ]:
1132/1409

In [ ]:
thresholds = np.linspace(0, 1, 21)
thresholds

In [ ]:
scores = []

for i in thresholds:
    churn_decision = (y_pred >= i)
    score = (y_val == churn_decision).mean()
    print('%.2f %.3f' %(i, score))
    scores.append(score)

In [ ]:
plt.plot(thresholds, scores)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(y_val, y_pred >= 0.6)

In [ ]:
scores = []

for i in thresholds:
    score = accuracy_score(y_val, y_pred >= i)
    print('%.2f %.3f' %(i, score))
    scores.append(score)

In [ ]:
from collections import Counter

In [ ]:
Counter(y_pred >= 1.0)

If you expect no one will churn and do nothing, you're 0.726 ~ 73% correct.

In [ ]:
actual_positive = (y_val == 1)
actual_negative = (y_val == 0)

In [ ]:
t = 0.5
predict_positive = (y_pred >= t)
predict_negative = (y_pred < t)

True positive

In [ ]:
actual_positive & predict_positive

In [ ]:
tp = (actual_positive & predict_positive).sum()
tp

True negative

In [ ]:
tn = (actual_negative & predict_negative).sum()
tn

False Positive and False Negative

In [ ]:
fp = (actual_negative & predict_positive).sum()
fn = (actual_positive & predict_negative).sum()
fp, fn

In [ ]:
confusion_matrix = np.array([
    [tn, fp],
    [fn, tp]
])

In [ ]:
confusion_matrix

In [ ]:
(confusion_matrix/ confusion_matrix.sum()).round(2)

Precision: Fraction of Positive predictions that are correct

In [ ]:
p = tp/(tp+fp)
p

Recall: Fraction of Correctly identified positives that are correct

In [ ]:
r = tp/(tp+fn)
r

When there is a class imbalance, Accuracy is not a good measure. That's why we use Precision, Recall.
Precision helps to see fraction of users who were wrongly given offers to prevent from churn = (1-p)
Recall helps to see fraction of users lost/ churned due to wrong identification = (1-r)

**ROC Curves**

TPR is the fraction of actual positives predicted correctly.
FPR is the fraction of actual negatives we got wrong.

In [ ]:
tpr = tp/(tp+fn)
fpr = fp/ (fp + tn)
round(tpr,3), round(fpr,3)

In [ ]:
thresholds = np.linspace(0, 1, 101)
scores = []
for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)

    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (actual_positive & predict_positive).sum()
    tn = (actual_negative & predict_negative).sum()
    fp = (actual_negative & predict_positive).sum()
    fn = (actual_positive & predict_negative).sum()

    scores.append((t, tp, fp, fn, tn))

In [ ]:
columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
df_scores = pd.DataFrame(scores, columns= columns)

In [ ]:
df_scores[::10]

In [ ]:
df_scores['tpr'] = df_scores.tp/(df_scores.tp+df_scores.fn)
df_scores['fpr'] = df_scores.fp/ (df_scores.fp + df_scores.tn)

In [ ]:
df_scores[::10]

In [ ]:
plt.plot(df_scores.threshold, df_scores.tpr, label = 'tpr')
plt.plot(df_scores.threshold, df_scores.fpr, label = 'fpr')
plt.legend

In [ ]:
np.random.seed(1)
y_rand = np.random.uniform(0, 1, len(y_val))

In [ ]:
y_rand.round(3)

In [ ]:
def tpr_fpr_dataframe(y_val, y_pred):

    thresholds = np.linspace(0, 1, 101)
    scores = []
    
    for t in thresholds:
        actual_positive = (y_val == 1)
        actual_negative = (y_val == 0)
    
        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)
    
        tp = (actual_positive & predict_positive).sum()
        tn = (actual_negative & predict_negative).sum()
        fp = (actual_negative & predict_positive).sum()
        fn = (actual_positive & predict_negative).sum()
    
        scores.append((t, tp, fp, fn, tn))

    columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
    df_scores = pd.DataFrame(scores, columns= columns)

    df_scores['tpr'] = df_scores.tp/(df_scores.tp+df_scores.fn)
    df_scores['fpr'] = df_scores.fp/ (df_scores.fp + df_scores.tn)

    return df_scores

In [ ]:
df_rand = tpr_fpr_dataframe(y_val, y_rand)

In [ ]:
df_rand[::10]

In [ ]:
plt.plot(df_rand.threshold, df_rand.tpr, label = 'tpr')
plt.plot(df_rand.threshold, df_rand.fpr, label = 'fpr')
plt.legend()

In [ ]:
num_neg = (y_val == 0).sum()
num_pos = (y_val == 1).sum()
num_neg, num_pos

In [ ]:
y_ideal = np.repeat([0,1], [num_neg, num_pos])
y_ideal

In [ ]:
y_ideal_pred = np.linspace(0,1,len(y_val))
y_ideal_pred.round(3)

In [ ]:
((y_ideal_pred >= 0.726) == y_ideal).mean()

In [ ]:
df_ideal = tpr_fpr_dataframe(y_ideal, y_ideal_pred)
df_ideal

In [ ]:
df_ideal[::10]

In [ ]:
plt.plot(df_ideal.threshold, df_ideal.tpr, label = 'tpr')
plt.plot(df_ideal.threshold, df_ideal.fpr, label = 'fpr')
plt.legend()

In [ ]:
plt.plot(df_scores.threshold, df_scores.tpr, label = 'tpr')
plt.plot(df_scores.threshold, df_scores.fpr, label = 'fpr')

#plt.plot(df_rand.threshold, df_rand.tpr, label = 'tpr')
#plt.plot(df_rand.threshold, df_rand.fpr, label = 'fpr')

plt.plot(df_ideal.threshold, df_ideal.tpr, label = 'tpr', color = 'black')
plt.plot(df_ideal.threshold, df_ideal.fpr, label = 'fpr', color = 'black')

In [ ]:
plt.figure(figsize=(5,5))

plt.plot(df_scores.fpr, df_scores.tpr, label = 'model')
plt.plot(df_rand.fpr, df_rand.tpr, label = 'rand')
plt.plot(df_ideal.fpr, df_ideal.tpr, label = 'ideal')

plt.xlabel('FPR')
plt.ylabel('TPR')

plt.legend()

In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, y_pred)

In [ ]:
plt.figure(figsize=(5,5))

plt.plot(fpr,tpr, label = 'model')
plt.plot([0,1], [0,1], linestyle = '--', label = 'rand')
#plt.plot(df_ideal.fpr, df_ideal.tpr, label = 'ideal')

plt.xlabel('FPR')
plt.ylabel('TPR')

plt.legend()

In [ ]:
from sklearn.metrics import auc

In [ ]:
auc(fpr, tpr)

In [ ]:
auc(df_scores.fpr, df_scores.tpr)

In [ ]:
auc(df_ideal.fpr, df_ideal.tpr)

In [ ]:
auc(df_rand.fpr, df_rand.tpr)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, y_pred)
auc(fpr, tpr)

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc_score(y_val, y_pred)

*AUC: Its the probability that a randomly selected positive example(Actual) has a higher score than a randomly selected negative example(Actual).*

In [ ]:
neg = y_pred[y_val==0]
pos = y_pred[y_val==1]

In [ ]:
import random

In [ ]:
pos_ind = random.randint(0, len(pos)-1)
neg_ind = random.randint(0, len(neg)-1)

In [ ]:
pos_ind, neg_ind

In [ ]:
pos[pos_ind] >= neg[neg_ind]

In [ ]:
n = 100000
success = 0

for i in range(n):

    pos_ind = random.randint(0, len(pos)-1)
    neg_ind = random.randint(0, len(neg)-1)

    if pos[pos_ind] >= neg[neg_ind]:
        success += 1
    
print(success/n)

In [ ]:
n = 10000

np.random.seed(1)
pos_ind = np.random.randint(0, len(pos), n)
neg_ind = np.random.randint(0, len(neg), n)

In [ ]:
pos_ind, neg_ind

In [ ]:
(pos[pos_ind] > neg[neg_ind]).mean()